# Column Transformer (Scikit-learn)

---

# 1. What is it? (Definition)

A **Column Transformer** is a preprocessing utility in **Scikit-learn** that allows you to apply **different transformations to different columns of a dataset**.

Instead of applying one preprocessing technique to the entire dataset, it lets you specify:

* Scale numerical columns
* Encode categorical columns
* Transform text columns
* Leave some columns unchanged
* Drop unnecessary columns

All within **one preprocessing pipeline**.

> **Definition**
>
> A Column Transformer is a Scikit-learn object that applies different preprocessing pipelines to different subsets of columns and combines their outputs into one transformed dataset.

---

# 2. Why do we need it?

Real-world datasets usually contain different types of features.

Example:

| Age | Salary | Gender | City  | Review      |
| --- | ------ | ------ | ----- | ----------- |
| 25  | 50000  | Male   | Delhi | "Very good" |

Notice:

* Age → Numeric
* Salary → Numeric
* Gender → Categorical
* City → Categorical
* Review → Text

Each type needs different preprocessing.

Without ColumnTransformer:

* Scale everything? ❌
* One-hot encode everything? ❌

Instead:

* Scale numbers
* Encode categories
* Vectorize text

simultaneously.

---

# 3. What problem does it solve?

Suppose your dataset has:

* Numerical features
* Categorical features
* Text features

Each requires different preprocessing.

Without ColumnTransformer:

```
Numeric
↓

StandardScaler

↓

Encoded manually

↓

Merged manually

↓

Risk of mistakes
```

Lots of manual work.

Problems:

* Repeated code
* Hard to maintain
* Easy to forget transformations
* Data leakage
* Column order confusion

ColumnTransformer automates everything.

---

# 4. What is the intuition behind it?

Imagine a hospital.

Different patients go to different departments.

```
Patient arrives

↓

Child
→ Pediatrics

↓

Eye problem
→ Ophthalmology

↓

Heart problem
→ Cardiology

↓

Results combined
```

Same idea.

Dataset arrives.

Different columns go to different transformers.

```
Dataset

↓

Numeric columns
→ StandardScaler

↓

Categorical columns
→ OneHotEncoder

↓

Text column
→ TfidfVectorizer

↓

Combine everything

↓

Final dataset
```

Every feature gets the treatment it needs.

---

# 5. How does it work? (Simple Steps)

Suppose dataset:

| Age | Salary | Gender | City  |
| --- | ------ | ------ | ----- |
| 25  | 50000  | Male   | Delhi |

### Step 1

Select numeric columns

```
Age
Salary
```

↓

Apply StandardScaler

---

### Step 2

Select categorical columns

```
Gender
City
```

↓

Apply OneHotEncoder

---

### Step 3

Transformer outputs

Numeric:

```
Scaled Age
Scaled Salary
```

Categorical:

```
Gender_Male
Gender_Female
City_Delhi
City_Mumbai
```

---

### Step 4

Concatenate

```
Scaled Age
Scaled Salary
Gender_Male
Gender_Female
City_Delhi
City_Mumbai
```

↓

Feed into ML model.

Everything happens automatically.

---

# 6. How does it work mathematically?

Suppose

Feature matrix

[
X =
\begin{bmatrix}
25 & 50000 & Male \
30 & 70000 & Female
\end{bmatrix}
]

Partition columns:

Numeric:

[
X_{num}
]

Categorical:

[
X_{cat}
]

Apply transformations independently.

Numeric

[
T_1(X_{num})
]

Example:

StandardScaler

[
z=\frac{x-\mu}{\sigma}
]

Categorical

[
T_2(X_{cat})
]

Example:

OneHotEncoder

Male →

[
[1,0]
]

Female →

[
[0,1]
]

Final output

[
X'
==

[T_1(X_{num}),
T_2(X_{cat})]
]

where

[
[\cdot]
]

means horizontal concatenation.

---

### Example

Original

| Age | Salary | Gender |
| --- | ------ | ------ |
| 25  | 50000  | Male   |

After scaling

[
[-0.8,,-1.2]
]

Gender

Male

↓

[
[1,0]
]

Final

[
[-0.8,,-1.2,;1,;0]
]

---

# 7. Important Concepts / Components

---

## A. Transformer

A transformer changes data.

Examples:

* StandardScaler
* MinMaxScaler
* OneHotEncoder
* OrdinalEncoder
* SimpleImputer
* TfidfVectorizer

A transformer follows:

```
fit()

learn parameters

↓

transform()

apply transformation

↓

fit_transform()

both together
```

---

## B. Columns

Specify which columns use which transformer.

Example

```
Age
Salary
```

or

```
["Age","Salary"]
```

---

## C. Transformer Tuple

Each transformation is defined as:

```
(name,
 transformer,
 columns)
```

Example

```python
("num", StandardScaler(), ["Age","Salary"])
```

---

## D. Remainder

Controls leftover columns.

Options:

```
drop
```

Default

Unspecified columns removed.

```
passthrough
```

Keep remaining columns unchanged.

---

## E. fit()

Learns preprocessing parameters.

Example

Mean

Variance

Categories

Missing values

etc.

---

## F. transform()

Uses learned parameters.

No relearning.

---

## G. fit_transform()

Performs

```
fit()

↓

transform()
```

together.

---

## H. Pipeline

Often used with ColumnTransformer.

```
ColumnTransformer

↓

Model
```

Entire workflow becomes reproducible.

---

# 8. Assumptions / Requirements

* Columns should be correctly identified.
* Appropriate transformer should match data type.
* Training and test data should have the same schema.
* Fit only on training data.
* Input should generally be tabular (Pandas DataFrame or NumPy array).

---

# 9. When should we use it?

Use ColumnTransformer when:

✅ Mixed numerical and categorical columns

✅ Different preprocessing needed

✅ Machine learning pipeline

✅ Production workflow

✅ Cross-validation

✅ Avoiding data leakage

Almost every structured ML project benefits from it.

---

# 10. When should we avoid it?

Avoid when:

* Entire dataset requires identical preprocessing
* Dataset contains only numerical columns
* Dataset contains only categorical columns
* Very small experiments where preprocessing is trivial
* Data is already fully preprocessed

---

# 11. Alternatives / Related Concepts

### Pipeline

Chains preprocessing and model.

```
Scaler

↓

Classifier
```

---

### FeatureUnion (older approach)

Combines outputs from multiple transformers but is generally less convenient for column-wise preprocessing.

---

### Manual preprocessing

```
StandardScaler

↓

OneHotEncoder

↓

Concatenate manually
```

Possible but not recommended.

---

### Pandas preprocessing

Use Pandas operations manually before training.

Good for exploration.

Not ideal for production pipelines.

---

# 12. How do we implement it?

```python
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), ["Age", "Salary"]),
        ("cat", OneHotEncoder(), ["Gender", "City"])
    ],
    remainder="drop"
)

X_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)
```

With a pipeline:

```python
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression())
])

pipeline.fit(X_train, y_train)
```

---

# 13. Important Parameters

## transformers

List of transformations.

```python
transformers=[
("num", StandardScaler(), num_cols),
("cat", OneHotEncoder(), cat_cols)
]
```

Most important parameter.

---

## remainder

```
drop
```

Remove unused columns.

```
passthrough
```

Keep unused columns.

---

## verbose_feature_names_out

Controls output feature names.

Default:

```
num__Age
```

Can be simplified by setting it to `False` in recent versions.

---

## sparse_threshold

Determines whether the combined output should be sparse or dense based on the density of transformed features.

---

## n_jobs

Parallel execution.

```
n_jobs=-1
```

Uses all CPU cores.

---

# 14. Common Mistakes

### Mistake 1

Scaling categorical columns.

❌

---

### Mistake 2

One-hot encoding numerical columns.

❌

---

### Mistake 3

Using fit() on test data.

Leads to data leakage.

---

### Mistake 4

Wrong column names.

Produces errors.

---

### Mistake 5

Different train/test columns.

Can cause transformation failures.

---

### Mistake 6

Forgetting `remainder="passthrough"` when you want to keep unspecified columns.

---

### Mistake 7

Performing preprocessing manually outside the pipeline, making cross-validation and deployment inconsistent.

---

# 15. Real-world Examples

## Banking

| Column | Transformation |
| ------ | -------------- |
| Age    | StandardScaler |
| Income | StandardScaler |
| Gender | OneHotEncoder  |
| City   | OneHotEncoder  |

---

## Healthcare

| Column         | Transformation |
| -------------- | -------------- |
| Age            | Scaler         |
| Blood Pressure | Scaler         |
| Disease        | OneHotEncoder  |

---

## E-commerce

| Column | Transformation    |
| ------ | ----------------- |
| Price  | Scaler            |
| Brand  | OneHotEncoder     |
| Review | TF-IDF Vectorizer |

---

## HR Analytics

| Column     | Transformation |
| ---------- | -------------- |
| Experience | Scaler         |
| Department | Encoder        |
| Education  | Encoder        |

---

# 16. How would I explain it in an interview?

> **Interview Answer (1–2 minutes):**
>
> "A ColumnTransformer is a Scikit-learn preprocessing tool that allows us to apply different transformations to different columns of a dataset in a single workflow. For example, we can scale numerical features using StandardScaler, encode categorical features using OneHotEncoder, and even vectorize text features, all at the same time. It automatically combines the transformed outputs into one feature matrix. This makes preprocessing cleaner, reduces manual coding, prevents data leakage when used inside a Pipeline, and ensures the same preprocessing is consistently applied during training, testing, cross-validation, and deployment."

---

# Revision Box

## One-line Definition

> **ColumnTransformer applies different preprocessing techniques to different columns and combines the results into one dataset.**

---

## Workflow

```
Dataset

↓

Select numeric columns
↓

StandardScaler

+

Select categorical columns
↓

OneHotEncoder

+

(Optional) Text columns
↓

TF-IDF

↓

Combine outputs

↓

Final feature matrix

↓

Machine Learning Model
```

---

## Key Formula

For feature matrix (X):

[
X' = \left[,T_1(X_{num}),; T_2(X_{cat}),; \cdots,; T_k(X_{subset_k}),\right]
]

where (T_i) is the transformer applied to a subset of columns, and the outputs are concatenated horizontally.

---

## Key Components

* **transformers** → List of `(name, transformer, columns)` tuples.
* **Transformer** → Performs preprocessing (e.g., `StandardScaler`, `OneHotEncoder`).
* **Columns** → Specifies which features each transformer operates on.
* **remainder** → `"drop"` or `"passthrough"` for unspecified columns.
* **fit()** → Learns preprocessing parameters.
* **transform()** → Applies learned transformations.
* **Pipeline** → Chains the preprocessor with an ML model.

---

## When to Use

* Mixed numerical and categorical data
* Different preprocessing for different features
* ML pipelines
* Cross-validation
* Production-ready preprocessing

---

## Common Mistakes

* Scaling categorical columns
* Encoding numerical columns
* Calling `fit()` on test data
* Wrong column selection
* Forgetting `remainder="passthrough"` when needed
* Doing preprocessing outside a `Pipeline`, leading to inconsistent training and inference


In [2]:
import numpy as np
import pandas as pd

In [3]:
# Import the SimpleImputer class from sklearn.impute.
# SimpleImputer is used to fill (impute) missing values (NaN)
# in a dataset using strategies such as:
#   - "mean"     -> replaces missing values with the column mean.
#   - "median"   -> replaces missing values with the column median.
#   - "most_frequent" -> replaces with the most frequent value.
#   - "constant" -> replaces with a user-defined constant value.
#
# Note:
# Your code contains an extra 'I' at the end, which is a syntax error.
# Correct import:
from sklearn.impute import SimpleImputer

# Import the OneHotEncoder class from sklearn.preprocessing.
# OneHotEncoder converts nominal categorical features
# (categories with NO natural order, e.g., Red, Blue, Green)
# into multiple binary (0/1) columns.
#
# Example:
# Color
# Red
# Blue
# Green
#
# becomes
# Red  Blue  Green
# 1      0      0
# 0      1      0
# 0      0      1
#
# This prevents machine learning algorithms from assuming
# any order among the categories.
from sklearn.preprocessing import OneHotEncoder

# Import the OrdinalEncoder class from sklearn.preprocessing.
# OrdinalEncoder converts ordinal categorical features
# (categories WITH a meaningful order)
# into integer values.
#
# Example:
# Low    -> 0
# Medium -> 1
# High   -> 2
#
# It is used only when the categories have a natural ranking.
# Otherwise, OneHotEncoder should be preferred.
from sklearn.preprocessing import OrdinalEncoder

In [5]:
# Read the csv file
df = pd.read_csv('covid_toy.csv')

In [12]:
# See the first 5 rows
df.head()

# Clearly, the age, and fever are numerical column
# while the gender, cough, and city are categorical column

# Gender and city are nominal catgeorical variables -> OHE
# Cough is ordinal -> Ordinal encoding

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [9]:
# See how many patients have which level of cough
df['cough'].value_counts()

# clearly, 62 had mild cough, and 38 had strong cough

,count
cough,
Mild,62
Strong,38


In [10]:
df['city'].value_counts()

,count
city,
Kolkata,32
Bangalore,30
Delhi,22
Mumbai,16


In [14]:
# Check the missing values in each column
df.isnull().sum()

# Clearly. fever has 10 missing values

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [17]:
# Import the train_test_split function from sklearn.model_selection.
# train_test_split is used to split the dataset into:
#   - Training data -> used to train the machine learning model.
#   - Testing data  -> used to evaluate the model on unseen data.
from sklearn.model_selection import train_test_split

# Split the dataset into training and testing sets.
#
# The function returns four objects in the following order:
#   X_train -> Training features (input variables used for training)
#   X_test  -> Testing features (input variables used for testing)
#   y_train -> Training target/output values
#   y_test  -> Testing target/output values
X_train, X_test, y_train, y_test = train_test_split(

    # df.drop(columns=['has_covid'])
    #
    # df represents the complete DataFrame.
    #
    # drop(columns=['has_covid']) removes the target column from the dataset,
    # leaving only the independent/input features.
    #
    # Parameter:
    # columns=['has_covid']
    #   - columns specifies which column(s) should be removed.
    #   - A list is used because multiple columns can also be dropped.
    #
    # This becomes the X (features) dataset.
    df.drop(columns=['has_covid']),

    # df['has_covid']
    #
    # Selects only the 'has_covid' column.
    #
    # This is the target (dependent variable) that the model
    # will learn to predict.
    #
    # This becomes the y (labels/target).
    df['has_covid'],

    # test_size=0.2
    #
    # Parameter:
    # test_size -> Specifies the proportion of data to keep for testing.
    #
    # Value:
    # 0.2 means 20% of the dataset is reserved for testing.
    # The remaining 80% is used for training.
    #
    # Example:
    # If the dataset has 100 rows:
    #   Training data = 80 rows
    #   Testing data  = 20 rows
    #
    # Keeping separate test data allows us to evaluate how well
    # the model performs on unseen data.
    test_size=0.2
)


In [18]:
# See the X_train
X_train

,age,gender,fever,cough,city
64,42,Male,104.0,Mild,Mumbai
52,47,Female,100.0,Strong,Bangalore
57,49,Female,99.0,Strong,Bangalore
37,55,Male,100.0,Mild,Kolkata
17,40,Female,98.0,Strong,Delhi
...,...,...,...,...,...
0,60,Male,103.0,Mild,Kolkata
78,11,Male,100.0,Mild,Bangalore
75,5,Male,102.0,Mild,Kolkata
81,65,Male,99.0,Mild,Delhi


1. Normal Method

In [19]:
# Without the Column transformer
 # Create an object (instance) of the SimpleImputer class.
#
# Since no parameter is provided, it uses the default parameter:
#
# strategy='mean'
# ----------------
# - Replaces missing (NaN) values with the mean (average)
#   of the column.
# - This strategy works only for numerical columns.
#
# Example:
# Fever column = [101, 102, NaN, 100]
# Mean = (101 + 102 + 100) / 3 = 101
# Missing value becomes:
# [101, 102, 101, 100]
si = SimpleImputer()


# Fit the imputer on the training data and then transform it.
#
# X_train[['fever']]
# ------------------
# - Selects only the 'fever' column from the training dataset.
# - Double square brackets [[ ]] are used so that the result
#   remains a DataFrame (2D), which sklearn expects.
#
# fit_transform()
# ---------------
# This performs two operations:
#
# 1. fit()
#    - Learns the required statistic from the training data.
#    - Since strategy='mean', it calculates the mean
#      of the 'fever' column.
#
# 2. transform()
#    - Replaces all missing values using the learned mean.
#
# Why use fit_transform() on training data?
# -----------------------------------------
# The model should learn preprocessing information
# only from the training dataset to avoid data leakage.
X_train_fever = si.fit_transform(X_train[['fever']])


# Apply the already learned imputation to the test data.
#
# X_test[['fever']]
# -----------------
# Selects only the 'fever' column from the test dataset.
#
# NOTE:
# -----
# The code below uses fit_transform(), but this is NOT the recommended practice.
#
# Correct approach:
# X_test_fever = si.transform(X_test[['fever']])
#
# Reason:
# -------
# - The imputer should NOT learn from the test data.
# - It should use the mean calculated from the training data.
# - Using fit_transform() here calculates a new mean from the
#   test set, causing data leakage and inconsistent preprocessing.
#
# Although your code works, transform() should be used instead.
X_test_fever = si.fit_transform(X_test[['fever']])


# Display the shape (dimensions) of X_train_fever.
#
# shape returns a tuple:
# (number_of_rows, number_of_columns)
#
# Example output:
# (80, 1)
#
# Meaning:
# - 80 rows
# - 1 column (fever)
#
# NOTE:
# -----
# Your code contains a typo:
# shapel
#
# Correct code:
# X_train_fever.shape
X_train_fever.shape

(80, 1)

In [20]:
# Create an object (instance) of the OrdinalEncoder class.
#
# OrdinalEncoder is used to convert ordinal categorical data
# (categories that have a meaningful order) into numerical values.
#
# Parameter:
# categories=[['Mild', 'Strong']]
# --------------------------------
# - categories specifies the exact order of the categories.
# - It must be a list of lists because you can encode multiple columns.
# - Here, there is only one column ('cough'), so one inner list is provided.
#
# Encoding performed:
# Mild   -> 0
# Strong -> 1
#
# Why specify the order manually?
# -------------------------------
# Without specifying categories, sklearn may determine the order
# automatically (usually alphabetical), which may not match the
# actual meaning of the data.
oe = OrdinalEncoder(categories=[['Mild', 'Strong']])


# Learn the mapping from the training data and convert the
# categorical values into numerical values.
#
# X_train[['cough']]
# ------------------
# Selects only the 'cough' column from the training dataset.
# Double square brackets [[ ]] keep the output as a DataFrame (2D),
# which is required by sklearn.
#
# fit_transform()
# ---------------
# Performs two operations:
#
# 1. fit()
#    - Learns the mapping between categories and numbers.
#    - Here, the mapping is already provided, so it simply validates
#      that the categories exist.
#
# 2. transform()
#    - Converts:
#        Mild   -> 0
#        Strong -> 1
#
# Why use fit_transform() on training data?
# -----------------------------------------
# The encoder should learn preprocessing information only from
# the training dataset.
X_train_cough = oe.fit_transform(X_train[['cough']])


# Convert the test data using the same encoder.
#
# X_test[['cough']]
# -----------------
# Selects the 'cough' column from the test dataset.
#
# NOTE:
# -----
# The code below uses fit_transform(), but this is NOT recommended.
#
# Correct code:
# X_test_cough = oe.transform(X_test[['cough']])
#
# Reason:
# -------
# - The encoder should NOT be fitted again on the test data.
# - It should use the mapping learned from the training data.
# - Using fit_transform() on the test data may cause inconsistent
#   preprocessing or data leakage.
X_test_cough = oe.fit_transform(X_test[['cough']])


# Display the shape (dimensions) of the transformed training data.
#
# shape returns:
# (number_of_rows, number_of_columns)
#
# Example:
# (80, 1)
#
# Meaning:
# - 80 rows
# - 1 encoded column ('cough')
#
# NOTE:
# The space before '.shape' does not affect execution.
X_train_cough.shape

(80, 1)

In [23]:
# Create an object (instance) of the OneHotEncoder class.
#
# OneHotEncoder is used to convert nominal categorical data
# (categories that have NO natural order) into multiple binary (0/1) columns.
#
# Parameters:
#
# drop='first'
# ------------
# - Drops the first category from each feature after encoding.
# - This helps avoid the Dummy Variable Trap (multicollinearity),
#   where one encoded column can be predicted from the others.
#
# Example:
# Gender
# Male
# Female
#
# Without drop='first':
# Male  Female
# 1       0
# 0       1
#
# With drop='first':
# Female
# 0   (Male)
# 1   (Female)
#
# sparse=False
# ------------
# - By default, OneHotEncoder returns a sparse matrix to save memory.
# - sparse=False converts the output into a normal NumPy array.
# - A NumPy array is easier to view and work with for beginners.
#
# NOTE:
# In newer versions of scikit-learn, sparse=False has been replaced by:
# sparse_output=False
ohe = OneHotEncoder(drop='first', sparse_output=False)


# Learn the categories from the training data and convert
# the categorical values into one-hot encoded columns.
#
# X_train[['gender', 'city']]
# ---------------------------
# Selects the 'gender' and 'city' columns from the training dataset.
#
# Double square brackets [[ ]] are used so that multiple columns
# are returned as a DataFrame (2D), which sklearn expects.
#
# fit_transform()
# ---------------
# Performs two operations:
#
# 1. fit()
#    - Learns all unique categories present in each column.
#
#      Example:
#      gender -> Male, Female
#      city   -> Delhi, Mumbai, Kolkata
#
# 2. transform()
#    - Converts each category into binary (0/1) columns.
#
# Example:
#
# gender   city
# Male     Delhi
#
# becomes (after drop='first'):
#
# Female  Mumbai  Kolkata
#   0        0        0
#
# Why use fit_transform() here?
# -----------------------------
# The encoder should learn categories only from the training data.
X_train_gender_city = ohe.fit_transform(X_train[['gender', 'city']])


# Apply the same encoding to the test data.
#
# X_test[['gender', 'city']]
# --------------------------
# Selects the same columns from the test dataset.
#
# NOTE:
# -----
# The code below uses fit_transform(), but this is NOT recommended.
#
# Correct code:
# X_test_gender_city = ohe.transform(X_test[['gender', 'city']])
#
# Reason:
# -------
# - The encoder should NOT learn categories again from the test data.
# - It should use the categories learned from the training data.
# - Using fit_transform() on the test data may create different
#   columns if some categories are missing or new categories appear,
#   leading to inconsistent feature sets and data leakage.
X_test_gender_city = ohe.fit_transform(X_test[['gender', 'city']])


# Display the shape (dimensions) of the encoded training data.
#
# shape returns:
# (number_of_rows, number_of_columns)
#
# The number of columns depends on:
# - Number of unique categories in 'gender'
# - Number of unique categories in 'city'
# - Whether drop='first' removes one category from each feature.
#
# Example:
# If:
# gender -> Male, Female
# city   -> Delhi, Mumbai, Kolkata
#
# Total encoded columns:
# (2 - 1) + (3 - 1) = 3 columns
#
# NOTE:
# The space before '.shape' does not affect execution.
X_train_gender_city.shape

(80, 4)

In [24]:
# Extract only the 'age' column from the training dataset.
#
# X_train.drop(columns=['gender', 'fever', 'cough', 'city'])
# -----------------------------------------------------------
# Removes the specified columns from the training dataset.
#
# Parameter:
# columns=['gender', 'fever', 'cough', 'city']
# --------------------------------------------
# - columns specifies which column(s) should be removed.
# - A list is used because multiple columns are being dropped.
#
# Since the dataset contains the following feature columns:
# ['gender', 'fever', 'cough', 'city', 'age']
#
# After dropping the first four columns,
# only the 'age' column remains.
#
# .values
# -------
# Converts the resulting Pandas DataFrame into a NumPy array.
#
# Why use .values?
# ----------------
# - Many scikit-learn models work directly with NumPy arrays.
# - It removes the DataFrame structure (row/column labels)
#   and keeps only the raw numerical values.
#
# Example:
# DataFrame:
#    age
# 0   25
# 1   40
#
# After .values:
# array([[25],
#        [40]])
X_train_age = X_train.drop(columns=['gender', 'fever', 'cough', 'city']).values


# Perform the same operation on the test dataset.
#
# X_test.drop(columns=['gender', 'fever', 'cough', 'city'])
# ----------------------------------------------------------
# Removes all columns except 'age' from the test dataset.
#
# .values converts the remaining DataFrame
# into a NumPy array.
#
# The same preprocessing is applied to both the
# training and testing datasets to maintain consistency.
X_test_age = X_test.drop(columns=['gender', 'fever', 'cough', 'city']).values


# Display the extracted age values.
#
# Since X_train_age is now a NumPy array,
# printing it will show only the numerical values.
#
# Example output:
# array([[25],
#        [42],
#        [36],
#        ...])
X_train_age

array([[42],
       [47],
       [49],
       [55],
       [40],
       [65],
       [ 5],
       [23],
       [60],
       [81],
       [19],
       [73],
       [42],
       [83],
       [24],
       [72],
       [10],
       [31],
       [48],
       [66],
       [64],
       [75],
       [46],
       [38],
       [65],
       [34],
       [65],
       [82],
       [18],
       [42],
       [81],
       [19],
       [ 8],
       [34],
       [49],
       [80],
       [82],
       [20],
       [10],
       [12],
       [26],
       [16],
       [70],
       [73],
       [68],
       [20],
       [24],
       [16],
       [14],
       [34],
       [ 5],
       [27],
       [59],
       [13],
       [33],
       [27],
       [64],
       [14],
       [50],
       [79],
       [83],
       [82],
       [71],
       [74],
       [25],
       [71],
       [47],
       [25],
       [56],
       [12],
       [17],
       [69],
       [83],
       [15],
       [19],
       [60],
       [11],

In [25]:
X_train_age.shape

(80, 1)

In [26]:
# Combine (concatenate) all the individually preprocessed feature arrays
# into one final training dataset.
#
# np.concatenate()
# ----------------
# Joins multiple NumPy arrays into a single NumPy array.
#
# Parameters:
#
# (
#     X_train_age,
#     X_train_fever,
#     X_train_gender_city,
#     X_train_cough
# )
# -------------------------
# A tuple containing the arrays to be combined.
#
# Each array represents one or more preprocessed features:
#
# X_train_age
# ----------------
# Contains the 'age' feature (already numerical).
#
# X_train_fever
# ----------------
# Contains the 'fever' feature after missing values
# have been filled using SimpleImputer.
#
# X_train_gender_city
# -----------------------
# Contains the One-Hot Encoded columns for
# 'gender' and 'city'.
#
# X_train_cough
# ----------------
# Contains the Ordinal Encoded 'cough' column.
#
# axis=1
# -------
# Specifies that the arrays should be joined horizontally
# (column-wise).
#
# axis=0 -> joins vertically (adds more rows)
# axis=1 -> joins horizontally (adds more columns)
#
# Since every array has the same number of rows,
# joining them column-wise creates one complete feature matrix.
#
# Example:
#
# X_train_age
# [[25]
#  [40]]
#
# X_train_fever
# [[101]
#  [100]]
#
# X_train_cough
# [[0]
#  [1]]
#
# After concatenation (axis=1):
#
# [[25 101 0]
#  [40 100 1]]
X_train_transformed = np.concatenate(
    (
        X_train_age,
        X_train_fever,
        X_train_gender_city,
        X_train_cough
    ),
    axis=1
)


# Perform the same concatenation for the test dataset.
#
# The test data must undergo exactly the same preprocessing
# as the training data so that both datasets have
# identical feature columns.
X_test_transformed = np.concatenate(
    (
        X_test_age,
        X_test_fever,
        X_test_gender_city,
        X_test_cough
    ),
    axis=1
)


# Display the shape (dimensions) of the final transformed
# training dataset.
#
# shape returns:
# (number_of_rows, number_of_columns)
#
# Example:
# (80, 5)
#
# Meaning:
# - 80 training samples (rows)
# - 5 total features (columns) after combining:
#     • age
#     • fever
#     • one-hot encoded gender columns
#     • one-hot encoded city columns
#     • ordinal encoded cough
#
# The exact number of columns depends on how many
# one-hot encoded columns were created.
X_train_transformed.shape

(80, 7)

In [27]:
X_train_transformed

array([[ 42.        , 104.        ,   1.        ,   0.        ,
          0.        ,   1.        ,   0.        ],
       [ 47.        , 100.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   1.        ],
       [ 49.        ,  99.        ,   0.        ,   0.        ,
          0.        ,   0.        ,   1.        ],
       [ 55.        , 100.        ,   1.        ,   0.        ,
          1.        ,   0.        ,   0.        ],
       [ 40.        ,  98.        ,   0.        ,   1.        ,
          0.        ,   0.        ,   1.        ],
       [ 65.        ,  99.        ,   1.        ,   0.        ,
          0.        ,   0.        ,   0.        ],
       [  5.        ,  98.        ,   0.        ,   0.        ,
          0.        ,   1.        ,   1.        ],
       [ 23.        ,  98.        ,   1.        ,   0.        ,
          0.        ,   1.        ,   1.        ],
       [ 60.        ,  99.        ,   0.        ,   0.        ,
          0.    

Up to this point, we manually preprocessed each feature based on its data type. We first split the dataset into training and testing sets, then filled missing values in the **`fever`** column using **`SimpleImputer`**. Next, we converted the **`cough`** column into numerical values using **`OrdinalEncoder`** and transformed the **`gender`** and **`city`** columns into binary columns using **`OneHotEncoder`**, while keeping the **`age`** column unchanged. Finally, we combined all these processed columns into a single dataset using **`np.concatenate()`**, which became the final input for our machine learning model.


2. Using the Column Transformer

In [33]:
# Import the ColumnTransformer class from sklearn.compose.
#
# ColumnTransformer allows us to apply different preprocessing
# techniques to different columns of the dataset in one place.
#
# It replaces the manual approach where we separately used
# SimpleImputer, OrdinalEncoder, OneHotEncoder, and then
# combined everything using np.concatenate().
from sklearn.compose import ColumnTransformer


# Create an object (instance) of the ColumnTransformer class.
transformer = ColumnTransformer(

    # transformers
    # ------------
    # A list of transformations to apply.
    #
    # Each transformation is written as a tuple:
    #
    # ('name', transformer_object, columns)
    #
    # where:
    # name               -> A unique name for the transformation.
    # transformer_object -> The preprocessing technique to apply.
    # columns            -> The column(s) on which the transformation
    #                       should be applied.
    transformers=[

        # First transformation (tnf1)
        #
        # 'tnf1'
        # ------
        # A custom name given to this transformation.
        # It can be any meaningful name.
        #
        # SimpleImputer()
        # ---------------
        # Replaces missing values in the selected column.
        #
        # Since no strategy is specified,
        # strategy='mean' is used by default.
        #
        # ['Fever']
        # ---------
        # Apply this imputer only to the 'Fever' column.
        ('tnf1', SimpleImputer(), ['fever']),


        # Second transformation (tnf2)
        #
        # 'tnf2'
        # ------
        # Custom name for this transformation.
        #
        # OrdinalEncoder(...)
        # ------------------
        # Converts ordinal categorical values into numbers.
        #
        # Parameter:
        # categories=[['Mild', 'Strong']]
        #
        # Specifies the order of categories:
        #
        # Mild   -> 0
        # Strong -> 1
        #
        # ['Cough']
        # ---------
        # Apply this encoder only to the 'Cough' column.
        ('tnf2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),


        # Third transformation (tnf3)
        #
        # 'tnf3'
        # ------
        # Custom name for this transformation.
        #
        # OneHotEncoder(...)
        # -----------------
        # Converts nominal categorical columns
        # into binary (0/1) columns.
        #
        # Parameters:
        #
        # sparse=False
        # ------------
        # Returns a NumPy array instead of a sparse matrix.
        #
        # NOTE:
        # In newer versions of scikit-learn,
        # use sparse_output=False instead.
        #
        # drop='first'
        # ------------
        # Drops the first category from each feature
        # to avoid the Dummy Variable Trap.
        #
        # ['Gender', 'City']
        # ------------------
        # Apply One-Hot Encoding to both
        # the Gender and City columns.
        ('tnf3', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'city'])
    ],

    # remainder='passthrough'
    # -----------------------
    # Specifies what to do with columns that are NOT mentioned
    # in the transformers list.
    #
    # 'passthrough'
    # -------------
    # Keeps the remaining columns unchanged.
    #
    # In this dataset, the 'Age' column is not listed above,
    # so it will automatically be included in the final output
    # without any preprocessing.
    #
    # If remainder='drop' were used instead,
    # the Age column would be removed.
    remainder='passthrough'
)

In [35]:
# Apply all the transformations defined in the ColumnTransformer
# to the training dataset and return the transformed data.
#
# transformer.fit_transform()
# ---------------------------
# Performs two operations:
#
# 1. fit()
#    -------
#    - Learns all the preprocessing information from the
#      training dataset.
#
#    Examples:
#    - SimpleImputer learns the mean of the 'Fever' column.
#    - OrdinalEncoder learns/validates the category order
#      (Mild -> 0, Strong -> 1).
#    - OneHotEncoder learns all unique categories present in
#      the 'Gender' and 'City' columns.
#
# 2. transform()
#    ------------
#    - Applies all the learned transformations to X_train.
#
#    Specifically:
#    - Imputes missing values in 'Fever'.
#    - Ordinal encodes 'Cough'.
#    - One-Hot encodes 'Gender' and 'City'.
#    - Passes the remaining column(s) (e.g., 'Age')
#      unchanged because remainder='passthrough'.
#
# X_train
# --------
# The original training dataset on which all preprocessing
# steps will be performed.
#
# .shape
# -------
# Returns the dimensions of the transformed dataset as:
# (number_of_rows, number_of_columns)
#
# Example output:
# (80, 5)
#
# Meaning:
# - 80 training samples (rows)
# - 5 processed feature columns (the exact number depends on
#   the number of categories created by OneHotEncoder).
transformer.fit_transform(X_train).shape

(80, 7)

In [36]:
transformer.transform(X_test).shape

(20, 7)